# [센서 실습 10] 일산화탄소 센서 – MQ-7

일산화탄소(CO) 센서 **MQ-7**의 출력을 ADC로 읽어 CO 농도를 계산하고, 난방기구 불완전 연소 상황에서 단계별 경보를 냅니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
MQ-7도 MQ-2처럼 **산화주석 반도체** 센서지만, 일산화탄소에 잘 반응하도록 만들어졌습니다.
- CO가 많을수록 센서 저항 Rs가 작아지고 출력 전압이 높아짐
- **히터 전압을 바꿔 가며** 측정하는 것이 특징: 5V로 60초 가열(청소) → 1.4V로 90초 유지 → **1.4V 구간 끝에서 측정**
- 일산화탄소는 색·냄새가 없어 사람이 알아차리지 못함 → 센서 경보가 생명을 지킴

| CO 농도 | 인체 영향 |
|---|---|
| 10 ppm | 다중이용시설 실내공기질 유지기준 |
| 50 ppm | 장시간 노출 시 두통 |
| 200 ppm | 2~3시간 내 두통, 어지러움 |
| 800 ppm | 45분 내 구토, 2시간 내 의식 잃음 |

### 부품 사양
| 항목 | 내용 |
|---|---|
| 감지 가스 | 일산화탄소(CO) |
| 감지 범위 | 20 ~ 2,000 ppm |
| 히터 | 5V(60초) / 1.4V(90초) 반복 |
| 출력 | 아날로그 (+ 디지털 D0) |
| 예열 | 최초 48시간 이상 권장 |

## 2. 라즈베리파이 연결 방법
**준비물**: MQ-7 모듈, MCP3008 ADC, 저항 10kΩ·20kΩ, 점퍼선

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| MCP3008 VDD, VREF (16, 15번) | 3.3V (1번 핀) | ADC 전원과 기준 전압 |
| MCP3008 AGND, DGND (14, 9번) | GND (6번 핀) |  |
| MCP3008 CLK (13번) | GPIO11 / SCLK (23번 핀) | SPI 클록 |
| MCP3008 DOUT (12번) | GPIO9 / MISO (21번 핀) | SPI 데이터 (ADC → Pi) |
| MCP3008 DIN (11번) | GPIO10 / MOSI (19번 핀) | SPI 데이터 (Pi → ADC) |
| MCP3008 CS (10번) | GPIO8 / CE0 (24번 핀) | 칩 선택 |
| MQ-7 VCC / GND | 5V / GND |  |
| MQ-7 A0 → 10kΩ → MCP3008 CH1 | CH1 ↔ GND 사이 20kΩ | 5V 출력을 2/3로 낮춤 |

> 💡 시중 MQ-7 모듈은 대부분 히터를 5V로 계속 켜 두는 단순 회로입니다. 정밀 측정에는 PWM으로 히터 전압을 바꾸는 회로가 필요하며, 이 실습에서는 '1.4V 구간 끝에서 읽은 값'을 받는다고 가정합니다.

**인터페이스 활성화**
- **SPI**: `sudo raspi-config` → Interface Options → **SPI** → Enable

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
MQ-2와 같은 순서로 계산하되, MQ-7 데이터시트의 **CO 곡선**(100 ppm에서 Rs/R0 = 1)을 사용합니다.

In [ ]:
VC, RL, R0 = 5.0, 10.0, 6.0       # 전원(V), 부하저항(kΩ), 100ppm CO에서의 기준 저항(kΩ)
DIVIDER = 1.5
M_CO, B_CO = -0.67, 1.34          # log10(Rs/R0) = m·log10(ppm) + b  (100ppm → Rs/R0 = 1)

def adc_to_ppm_mq7(adc_value, vref=3.3):
    v_out = max(adc_value * vref * DIVIDER, 1e-3)
    rs = (VC - v_out) / v_out * RL
    return 10 ** ((np.log10(rs / R0) - B_CO) / M_CO)

for a in (0.12, 0.25, 0.40, 0.55):
    print(f"ADC {a:.2f} → {adc_to_ppm_mq7(a):7.1f} ppm")

### 4-2. 실제 센서 클래스
ADC 1번 채널을 읽어 CO 농도를 계산합니다. (가스 센서와 함께 쓰는 경우를 생각해 다른 채널을 사용)

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealMQ7:
    def __init__(self, channel=1):
        from gpiozero import MCP3008
        self.adc = MCP3008(channel=channel)

    def read(self):
        v = self.adc.value
        return {"adc": round(v, 4), "co_ppm": round(adc_to_ppm_mq7(v), 1)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 상황 | 겨울철 캠핑 텐트·창고에서 난로 사용, 3시간 | 실제 사고가 잦은 상황 |
| 평상시 | 3 ~ 5 ppm | 일반 실내 |
| 불완전 연소 | 40분부터 산소 부족으로 점점 증가, 최대 약 250 ppm | 밀폐 공간 난로 |
| 조치 | 120분에 난로 끄고 환기 → 감소 | 경보 후 대응 |
| 잡음 | ±5% | 센서 특성 편차 |

In [ ]:
class VirtualMQ7:
    def __init__(self, dt=150, seed=10):          # 히터 1주기 = 150초마다 1회 측정
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    @staticmethod
    def ppm_to_adc(ppm, vref=3.3):
        rs = 10 ** (M_CO * np.log10(ppm) + B_CO) * R0
        return VC * RL / (RL + rs) / DIVIDER / vref

    def read(self):
        m = self.t / 60
        ppm = 4.0
        if 40 <= m < 120:
            ppm += 250 / (1 + np.exp(-(m - 85) / 10))            # 불완전 연소로 서서히 증가
        elif m >= 120:
            ppm += 250 / (1 + np.exp(-35 / 10)) * np.exp(-(m - 120) / 10)   # 환기
        ppm *= np.exp(self.rng.normal(0, 0.05))
        adc = round(float(self.ppm_to_adc(ppm)), 4)
        self.t += self.dt
        return {"adc": adc, "co_ppm": round(adc_to_ppm_mq7(adc), 1)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualMQ7(dt=150)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
히터 1주기(150초)마다 한 번씩, 3시간 동안 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 72               # 측정 횟수
INTERVAL = 150               # 측정 간격(초)

sensor = RealMQ7() if USE_REAL_SENSOR else VirtualMQ7(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
| CO 농도 | 단계 | 조치 |
|---|---|---|
| 10 ppm 미만 | 정상 | – |
| 10 ~ 50 ppm | 주의 | 환기 권고 |
| 50 ~ 200 ppm | 경보 | 난방기 끄고 환기 |
| 200 ppm 이상 | 위험 | 즉시 대피 |

In [ ]:
df["min"] = df["time_s"] / 60
def stage(p):
    return "정상" if p < 10 else "주의" if p < 50 else "경보" if p < 200 else "위험"
df["stage"] = df["co_ppm"].apply(stage)
prev = None
for _, r in df.iterrows():
    if r["stage"] != prev:
        print(f"{r['min']:6.1f}분  {r['co_ppm']:6.1f} ppm  → {r['stage']}")
        prev = r["stage"]
print("\n단계별 시간(분):", (df["stage"].value_counts() * INTERVAL / 60).round(1).to_dict())

## 8. 시각화

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(df["min"], df["co_ppm"], "k-o", ms=3, lw=1.5, label="CO 농도")
for y, name in [(10, "주의"), (50, "경보"), (200, "위험")]:
    plt.axhline(y, color="gray", ls="--", lw=0.8); plt.text(181, y, name, va="center")
plt.title("MQ-7 일산화탄소 농도"); plt.xlabel("시간 (분)"); plt.ylabel("ppm")
plt.grid(alpha=0.3); plt.legend(loc="upper left"); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "일산화탄소센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "MQ-7",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/co", payload)
# client.disconnect()

## 직접 해 보기
- '주의' 단계에 처음 들어간 뒤 '경보'까지 몇 분이 걸렸나요? 이 시간이 대피에 왜 중요할까요?
- 측정 간격을 30초로 바꾸면 어떤 장단점이 있을까요? (히터 주기와 비교)
- MQ-2 노트북과 비교해 두 센서의 변환식에서 무엇이 같고 무엇이 다른지 정리해 보세요.